# GSB 5544 — Topic 10.2: Regularization — Ridge, LASSO, Tuning Parameters and Cross-Validation — SOLUTION
*A 15–20 minute think-and-discuss opener before the penalized-regression walkthrough and PA 10.2. One running
example, five questions, no derivations.*

## How to use this

**The running example.** A regional coffee chain has 100 locations. For each store we know **ten things** about
the store and its neighborhood — foot traffic, ad spend, store size, seats, parking spaces, neighborhood income,
competitor count, social-media posts, average temperature, manager tenure — and **last week's sales**. The chain
is opening new stores and wants a **sales forecast from a store's characteristics alone**. 60 stores are the
training data; 40 are locked away as a test set. (This is the example on the interactive page
[*Shrink to Fit*](https://gato365.github.io/gsb5544_instructor_learn_prep/week_10/shrink_to_fit.html), where the sliders show every number in this notebook moving; the questions below are
also in the page's *Think · Discuss · Reveal* box.)

**The rhythm.** For each question: think for a minute, discuss with a neighbour for two, then the instructor
reveals the answer. Five questions, three to four minutes each.

**The three objectives**, so they are in front of us the whole time. Each adds a term to the one before it.

$$\text{Least squares:}\quad \min_{\beta_0,\boldsymbol\beta} \underbrace{\sum_{i=1}^{n} (y_i - \hat y_i)^2}_{\text{prediction error on the training rows}}$$

$$\text{Ridge:}\quad \min_{\beta_0,\boldsymbol\beta} \Big[\underbrace{\sum_{i=1}^{n} (y_i - \hat y_i)^2}_{\text{prediction error}} + \underbrace{\lambda \sum_{j=1}^{p} \beta_j^2}_{\text{penalty: squared coefficients}}\Big]$$

$$\text{LASSO:}\quad \min_{\beta_0,\boldsymbol\beta} \Big[\underbrace{\sum_{i=1}^{n} (y_i - \hat y_i)^2}_{\text{prediction error}} + \underbrace{\lambda \sum_{j=1}^{p} |\beta_j|}_{\text{penalty: absolute coefficients}}\Big]$$

| symbol | what it is |
|---|---|
| $y_i$, $\hat y_i$ | actual and predicted weekly sales for training store $i$; $n = 60$ |
| $\beta_1, \dots, \beta_p$ | the **coefficients**, one per predictor ($p = 10$); learned by fitting |
| $\beta_0$ | the intercept — **not penalized**; it only sets the overall level of sales |
| $\lambda \ge 0$ | the **tuning parameter**: how heavily coefficient size is charged. Not learned by fitting; chosen by cross-validation |

> **About `alpha`.** scikit-learn calls $\lambda$ `alpha`, and it scales the error term differently for different
> models (`Lasso` and `ElasticNet` divide it by $2n$; `Ridge` does not). So `alpha=1` is **not** the same strength of
> penalty for ridge and for LASSO, and a good `alpha` on one data set says nothing about another. Compare penalty
> strengths by their cross-validation curves, never by the number itself.

---
**Q1.** Plain linear regression first. We fit weekly sales ≈ intercept + coefficients × the ten store characteristics to the 60 training stores. What does least squares minimize — and why can a model that fits those 60 stores very well still forecast a *new* store poorly? (Three of the ten predictors — seats, parking, store size — all measure how big the store is, and two may be pure noise.)

**Answer:** Least squares picks the intercept and coefficients that make the **sum of squared training errors** as small as possible, judged only on the 60 stores it saw. So it bends to whatever is peculiar about them: when predictors overlap, many coefficient combinations fit almost equally well (a big positive coefficient on seats cancelling a big negative one on store size), and least squares picks whichever suits *this* sample — a different 60 stores can flip the signs. Noise predictors get coefficients for the same reason. Low training error is a fact about the past sample; forecasting is a claim about new stores, and last week's cross-validated test error is the number that speaks to that.

---
**Q2.** The penalty. Regularization changes what we minimize to **training error + a penalty on coefficient size**. Why would deliberately accepting a slightly *worse* fit to the 60 stores make forecasts for new stores *better*, and what decides how the two terms trade off?

**Answer:** Part of what unrestrained coefficients capture is signal and part is sample noise; pulling them toward zero gives up a little signal (more training error) but throws away a lot of noise, so predictions vary less from sample to sample — usually a favourable trade, though never a guaranteed one. The exchange rate is the tuning parameter $\lambda$ (`alpha` in scikit-learn): a larger $\lambda$ makes coefficient size more expensive, so the solution shrinks. The intercept is left out of the penalty — it only sets the overall level of sales, and shrinking it would bias every forecast.

*Follow-up:* Training error can only get worse as $\lambda$ grows; validation error falls first and rises later. That shape is the whole argument.

---
**Q3.** Two penalties. Ridge charges $\lambda \sum_j \beta_j^2$ (squares); LASSO charges $\lambda \sum_j |\beta_j|$ (absolute values). Why does ridge shrink every coefficient but almost never reach zero, while LASSO can set some coefficients to *exactly* zero? And why must the predictors be standardized before either penalty is applied?

**Answer:** Squaring makes big coefficients very expensive and small ones nearly free, so ridge trims the largest hardest and the last step from 0.1 to 0 saves only 0.01 — almost never worth taking. Under absolute values that last step saves as much as any other, so a predictor that is not earning its keep lands on exactly zero and leaves the model: LASSO shrinks *and* selects, which gives a smaller, cheaper, more explainable model, though among overlapping predictors it keeps one somewhat arbitrarily (ridge shares the effect across the group). Both penalties treat a coefficient of 500 as "big" whatever the variable, yet sales per square foot and sales per year of tenure live on different scales; **standardizing** each predictor puts every coefficient in dollars per standard deviation so the penalty is fair — fitted inside each cross-validation training fold, or the validation rows leak into it.

*Follow-up:* A LASSO zero on `store size` while `seats` is kept does **not** mean size is irrelevant, let alone causally inert: LASSO chose a stand-in. Regularization is a forecasting tool, not a hypothesis test. Large coefficients are not automatically a problem either; their size depends on units until the predictors are standardized.

---
**Q4.** The tuning parameter. What does the model become when $\lambda = 0$, when it is small, and when it is enormous? Here are six candidates and the 5-fold cross-validation MSE each produced on the 60 training stores (hypothetical, millions of dollars²):

| $\lambda$ | 0 | 0.1 | 1 | 10 | 100 | 1000 |
|---|---|---|---|---|---|---|
| CV MSE | 9.8 | 9.1 | 8.4 | 8.2 | 9.6 | 15.0 |

Which would you choose, and why could training MSE not have made this choice?

**Answer:** At $\lambda = 0$ the penalty vanishes and we have ordinary least squares; small $\lambda$ shrinks gently; as it grows the coefficients head to zero (LASSO drops predictors one by one) and the forecast collapses toward the average sales of the training stores — a model that ignores the store. The coefficients are **learned** by fitting for a given $\lambda$; $\lambda$ itself is **tuned** from outside by comparing candidates on validation data. Pick $\lambda = 10$: lowest CV error, and the curve has the expected shape — the strongest penalty (15.0) is clearly not the best. Training MSE would have chosen 0 every time, since any penalty worsens the fit to the training stores; and because 8.4 and 8.2 are within the noise of five folds, $\lambda = 1$ is defensible too.

*Follow-up:* scikit-learn scales the error term differently across models (`Lasso` divides by $2n$, `Ridge` does not), so `alpha = 1` is not the same penalty for ridge and LASSO, nor across data sets. The number means nothing on its own; the CV curve does.

---
**Q5.** Grid search, the test set, and the whole picture. `GridSearchCV` is handed a list of candidate $\lambda$ values. What does it do with them? The chain locked 40 stores away before we began — if cross-validation already scores each candidate on unseen stores, why keep those 40 untouched until the end? Finish by explaining to a manager how regularization, ridge/LASSO, the tuning parameter and cross-validation work together to forecast a store the chain has never seen.

**Answer:** Grid search runs the *same* K-fold cross-validation — same folds — for every candidate, records each mean validation score, keeps the best, and refits it on all the training rows; it is the table above, automated. The best of several candidates is partly the luckiest, so its validation score is a little optimistic — the 40 stores took no part in any decision, so the one error computed on them after refitting is the honest estimate; open them once, report, and do not re-tune. To the manager: we fit a linear model but charge a price for large coefficients, so it stops memorizing quirks of our sample (regularization); ridge charges by squared size and shrinks everything a little, LASSO charges by absolute size and can drop predictors, giving a simpler model; how heavy the charge should be is a tuning parameter the fit cannot tell us, so we try a grid of values and let cross-validation on the training stores vote for the one that forecasts held-out stores best; we refit with that value, check it once on the stores we locked away, and that model — with standardized inputs — forecasts the new store.

---
## Misconceptions to name out loud

| students often think… | but… |
|---|---|
| "A big coefficient means that predictor is important / is a problem." | Coefficient size depends on the predictor's units until the predictors are standardized. Size is only comparable after that, and even then size is about the fit, not about importance in any causal sense. |
| "Regularization makes the model better." | It trades a little fit for less variance. Usually that helps on new data; sometimes (clean data, few predictors, many rows) it does nothing or hurts. Cross-validation tells you which. |
| "More penalty is safer." | Too much penalty flattens the model toward "predict the average sales": the CV curve rises again. The best penalty is the one with the lowest validation error, often moderate. |
| "Ridge and LASSO both do variable selection." | Ridge shrinks toward zero but essentially never reaches it; LASSO can land exactly on zero. Only LASSO (and elastic net) removes predictors. |
| "LASSO set `store size` to zero, so store size doesn't matter." | It means store size was not needed *for prediction given the other predictors* — very likely because `seats` carries the same information. It is not evidence of no effect and certainly not of no causal effect. |
| "Standardize once, then cross-validate." | The scaler is fitted; fit it on all rows and every validation fold has leaked its mean and spread into training. Put the scaler in the pipeline so it is refitted inside each training fold. |

---
## One tiny demonstration (optional, two minutes)

A simulated coffee chain in eight lines: five real predictors, one correlated copy (`seats` tracks `size`), one
pure-noise column. Standardize, then fit least squares, ridge and LASSO at one penalty and compare the
coefficients. Watch which method shrinks and which zeros. (The `alpha` values differ between the two models on
purpose — see the note above.)

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(5544)
n = 60
stores = pd.DataFrame({
    "traffic": rng.normal(800, 200, n),  "ads": rng.normal(300, 100, n),
    "size":    rng.normal(1800, 400, n), "income": rng.normal(65, 12, n),
    "competitors": rng.integers(0, 6, n),
})
stores["seats"]  = stores["size"] / 30 + rng.normal(0, 4, n)      # a correlated copy of size
stores["tenure"] = rng.normal(5, 3, n)                               # pure noise
sales = (6 * stores["traffic"] + 5 * stores["ads"] + 2 * stores["size"] + 60 * stores["income"]
         - 400 * stores["competitors"] + rng.normal(0, 2800, n))

fits = {
    "least squares": make_pipeline(StandardScaler(), LinearRegression()),
    "ridge":         make_pipeline(StandardScaler(), Ridge(alpha=10)),
    "LASSO":         make_pipeline(StandardScaler(), Lasso(alpha=300)),
}
coefs = pd.DataFrame({name: pipe.fit(stores, sales)[-1].coef_ for name, pipe in fits.items()}, index=stores.columns)
coefs.round(0) + 0.0        # + 0.0 turns LASSO's "-0.0" into a plain 0

,least squares,ridge,LASSO
traffic,1667.0,1418.0,1289.0
ads,449.0,342.0,87.0
size,347.0,219.0,0.0
income,1401.0,1200.0,1042.0
competitors,-753.0,-637.0,-409.0
seats,234.0,223.0,106.0
tenure,-141.0,-157.0,0.0


✅ **What to notice.** All three agree on the strong predictors (traffic, ads, income, competitors). Least squares
splits the "how big is the store" effect between `size` and `seats` in some arbitrary way and gives `tenure` a
non-zero coefficient it does not deserve; ridge shrinks every coefficient and shares the size effect across the
pair; LASSO shrinks too and sets some coefficients exactly to zero. Which of `size` and `seats` LASSO keeps is a
stand-in choice, not a finding. The penalty values here were set by hand; in the walkthrough and the PA they are
chosen by `GridSearchCV`.

---
## Where this goes next

1. **The sliders.** [*Shrink to Fit*](https://gato365.github.io/gsb5544_instructor_learn_prep/week_10/shrink_to_fit.html): the same coffee chain; drag the penalty and watch the coefficients,
   the training and CV error curves, and the coefficient paths; run the grid search fold by fold; open the test
   set once at the end.
2. **The code walkthrough** (*Penalized Regression: Ridge and LASSO*, the first part of the PA 10.2 notebook):
   the Ames houses with *all* 67 predictors, where plain least squares with polynomial terms falls apart under
   cross-validation; then `Ridge`, `Lasso` and `ElasticNet` in a pipeline, counting zero coefficients, and
   `GridSearchCV` over `alpha` (and `l1_ratio`). Its twelve **DISCUSS** prompts are the questions above in Ames
   clothing — Q2–Q4 of this notebook answer them.
3. **PA 10.2.** The loans data again, predicting `interest_rate` — this time with ridge, LASSO and elastic net on
   many predictors, at least one parameter tuned with `GridSearchCV`. Bring the ideas: standardize inside the
   pipeline, choose `alpha` by cross-validation, read a zero coefficient as "not needed given the others", and
   keep the test estimate separate from the tuning.

Course site: [https://gato365.github.io/gsb5544_instructor_learn_prep/](https://gato365.github.io/gsb5544_instructor_learn_prep/).